## Import All The Libraries


In [1]:
%pip install torcheval 

Note: you may need to restart the kernel to use updated packages.


In [2]:
from sklearn.preprocessing import StandardScaler  
from sklearn.model_selection import train_test_split  
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import copy
import torch
import tqdm

# Format and Adjust Data

In [3]:
df = pd.read_csv("dataset.csv")

# One Hot Encoding the 'stock' column
df = pd.get_dummies(df, columns=['stock'])

# remove data leaking columns
df.pop('next_weeks_open') 
df.pop('percent_change_next_weeks_price')

# Replace missing values
cols = ['previous_weeks_volume','percent_change_volume_over_last_wk']
df[cols] = df[cols].replace('?',0)

# Handle Date Column's String Type 
df['date'] = pd.to_datetime(df['date'])
# Distribute Date into new columns
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
# Pop old 'date' column, we have month,day,year to represent its values
temp_date = df.pop('date')

# Convert all dollar values from string into floats
df = df.replace('\$','',regex=True).astype(float)

# Shift label column to rightmost for eash of indexing 
y_col = df.pop('next_weeks_close')
df.insert(len(df.columns),'next_weeks_close',y_col)

<>:24: SyntaxWarning: invalid escape sequence '\$'
<>:24: SyntaxWarning: invalid escape sequence '\$'
/var/folders/16/22h86xpd7j33w8md_8shh5gh0000gn/T/ipykernel_4161/3559829027.py:24: SyntaxWarning: invalid escape sequence '\$'
  df = df.replace('\$','',regex=True).astype(float)


## Scale the dataset

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import numpy as np

# Convert dataframe to numpy
data = df.to_numpy()

# Split into features and labels
X = data[:, 0:len(df.columns)-1]
y = data[:, len(df.columns)-1]
y_1 = np.reshape(y, (y.shape[0], 1))

# Standardize features and labels
x_scalar = StandardScaler()
y_scalar = StandardScaler()

X_scaled = x_scalar.fit_transform(X)
y_scaled = y_scalar.fit_transform(y_1)

# FIRST split: training+validation and test (e.g., 80% train+val, 20% test)
X_train_val, X_test, y_train_val, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, shuffle=True, random_state=42)

# SECOND split: from training+validation, split into train and validation (e.g., 80% train, 20% val of that 80%)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, shuffle=True, random_state=42)

# Print shapes
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

# Convert all splits to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

X_train shape: (480, 44)
X_val shape: (120, 44)
X_test shape: (150, 44)
y_train shape: (480, 1)
y_val shape: (120, 1)
y_test shape: (150, 1)


# Create NN Models 

In [5]:
model1 = nn.Sequential(
    nn.Linear(44, 64),
    nn.ELU(),
    nn.Linear(64, 32),
    nn.ELU(),
    nn.Linear(32, 16),
    nn.ELU(),
    nn.Linear(16, 1)
)

model2 = nn.Sequential(
    nn.Linear(44, 64),
    nn.ELU(),
    nn.Dropout(0.3),  
    nn.Linear(64, 32),
    nn.ELU(),
    nn.Dropout(0.3),
    nn.Linear(32, 16),
    nn.ELU(),
    nn.Dropout(0.3),
    nn.Linear(16, 1)
)

model3 = nn.Sequential(
    nn.Linear(44, 32),
    nn.ELU(),
    nn.Dropout(0.3),  
    nn.Linear(32, 15),
    nn.ELU(),
    nn.Dropout(0.3),
    nn.Linear(16, 1)
)

models = [model1,model2,model3]

# Train Models

In [10]:
def train_model(model, X_train, y_train, X_val, y_val, l2_val):
    n_epochs = 500
    batch_size = 32
    loss_fn = nn.MSELoss()  
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=l2_val)
    batch_start = torch.arange(0, len(X_train), batch_size)
    
    # For tracking performance
    best_val_mse = float('inf')
    best_weights = None
    train_history = []
    val_history = []
    
    # For early stopping
    patience = 30
    patience_counter = 0
    
    for epoch in range(n_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        batches = 0
        
        with tqdm.tqdm(batch_start, unit="batch", mininterval=0, disable=True) as bar:
            bar.set_description(f"Epoch {epoch}")
            for start in bar:
                # take a batch
                end = min(start + batch_size, len(X_train))
                X_batch = X_train[start:end]
                y_batch = y_train[start:end]
                
                # Skip batches that are too small
                if len(X_batch) < 2:
                    continue
                
                # forward pass
                y_pred = model(X_batch)
                loss = loss_fn(y_pred, y_batch)
                
                # backward pass
                optimizer.zero_grad()
                loss.backward()
                
                # update weights
                optimizer.step()
                
                # accumulate loss
                train_loss += float(loss)
                batches += 1
                
                # print progress
                bar.set_postfix(mse=float(loss))
        
        # Calculate average training loss for this epoch
        avg_train_loss = train_loss / max(1, batches)
        train_history.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = float(loss_fn(val_pred, y_val))
            val_history.append(val_loss)
            
            # Print progress every 100 epochs or at the end
            if epoch % 100 == 0 or epoch == n_epochs - 1:
                print(f"Epoch {epoch}: Train MSE: {avg_train_loss:.6f}, Val MSE: {val_loss:.6f}")
            
            # Save model if validation loss improves
            if val_loss < best_val_mse:
                best_val_mse = val_loss
                best_weights = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                
            # Early stopping
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
    
    # Restore best model
    if best_weights is not None:
        model.load_state_dict(best_weights)
    
    return {
        'model': model,                    # The trained model
        'train_history': train_history,    # Training loss history
        'val_history': val_history,        # Validation loss history
        'final_train_loss': train_history[-1],  # Final training loss
        'final_val_loss': val_history[-1],      # Final validation loss
        'best_val_loss': best_val_mse,     # Best validation loss
        'n_epochs_trained': len(train_history),  # Actual number of epochs trained
        'weight_decay': l2_val             # Weight decay used
    }

# Create Evaluation Model 

In [ ]:
from torcheval.metrics import R2Score
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name="Model"):

    # Set model to evaluation mode
    model.eval()
    metrics = {}
    
    with torch.no_grad():
        # Get predictions on training data
        y_train_pred = model(X_train)
        train_mse = float(nn.MSELoss()(y_train_pred, y_train))
        train_rmse = np.sqrt(train_mse)
        
        # Get predictions on test data
        y_test_pred = model(X_test)
        test_mse = float(nn.MSELoss()(y_test_pred, y_test))
        test_rmse = np.sqrt(test_mse)

        
        # Calculate R^2 (coefficient of determination)
        # Compute Training R^2
        r2_train_metric = R2Score()
        r2_train_metric.update(y_train_pred,y_train)
        r2_train = r2_train_metric.compute()


        # Compute Test R^2
        r2_test_metric = R2Score()
        r2_test_metric.update(y_test_pred,y_test)
        r2_test = r2_test_metric.compute()
        
        # Store all metrics
        metrics = {
            'model_name': model_name,
            'train_mse': train_mse,
            'train_rmse': train_rmse,
            'train_r2': r2_train,
            'test_mse': test_mse,
            'test_rmse': test_rmse,
            'test_r2': r2_test,
            'overfitting_ratio': train_rmse / test_rmse if test_rmse > 0 else float('inf')
        }

    return metrics
    # # Calculate percentage errors (RMSPE)
    # train_squared_perc_errors = ((y_train_orig - y_train_pred_orig) / y_train_orig) ** 2
    # train_rmspe = np.sqrt(np.mean(train_squared_perc_errors))
    
    # test_squared_perc_errors = ((y_test_orig - y_test_pred_orig) / y_test_orig) ** 2
    # test_rmspe = np.sqrt(np.mean(test_squared_perc_errors))
    
    # # # Calculate MAPE (Mean Absolute Percentage Error)
    # train_mape = np.mean(np.abs((y_train_orig - y_train_pred_orig) / y_train_orig))
    # test_mape = np.mean(np.abs((y_test_orig - y_test_pred_orig) / y_test_orig))
            # 'train_rmspe': train_rmspe,
            # 'train_mape': train_mape,
            # 'test_rmspe': test_rmspe,
            # 'test_mape': test_mape,
        
        # Convert predictions back to original scale
        # y_train_pred_orig = y_scalar.inverse_transform(y_train_pred.numpy())
        # y_train_orig = y_scalar.inverse_transform(y_train.numpy())
        
        # y_test_pred_orig = y_scalar.inverse_transform(y_test_pred.numpy())
        # y_test_orig = y_scalar.inverse_transform(y_test.numpy())

# Test different regularization values and models

In [12]:
# Function to run multiple regularization experiments
def run_regularization_experiments(models, weight_decay_values):

    results = {}

    # Iterate through all 3 models
    for i in range(len(models)):
        model_name = "model"+str(i)
        # Test each L2 value for each 
        for weight_decay in weight_decay_values:
            print(f"\n\n=== Training {model_name} with weight_decay={weight_decay} ===")
            
            # Train the model
            train_model(models[i], X_train, y_train,X_val,y_val,weight_decay)
            
            print(f"\n\n=== Evaluating model with weight_decay={weight_decay} ===")
            # Evaluate the model
            metrics = evaluate_model(models[i], X_train, y_train, X_test, y_test, y_scalar, model_name)
            
            # Store results
            results[model_name] = metrics
        
    return results


# Run tests

In [13]:

weight_decay_values = [0, 0.0001, 0.001, 0.01, 0.1, 1.0]

res = run_regularization_experiments(models[:1], weight_decay_values[:1])


print("RESULTS--------------------------------------------------")


data = res['model0']

for key,val in data.items():
    print(key,":",val)
    




=== Training model0 with weight_decay=0 ===
Epoch 0: Train MSE: 0.794411, Val MSE: 0.504604
Early stopping at epoch 91


=== Evaluating model with weight_decay=0 ===
RESULTS--------------------------------------------------
model_name : model0
train_mse : 0.0016197493532672524
train_rmse : 0.04024610979047854
train_r2 : tensor(0.9984)
test_mse : 0.0024212554562836885
test_rmse : 0.04920625423951399
test_r2 : tensor(0.9976)
overfitting_ratio : 0.8179063904067665
